# Cloud and HPC Resources for Heavy Computational Science
### Free, Low-Cost, and National-Scale Computing for Computational Toxicology and Drug Discovery

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

> **Note:** This tutorial assumes you are reading 'FREE cloud resources' — covering no-cost and
> subsidised platforms for academic computational scientists, from Google Colab (completely free)
> to NSF ACCESS national supercomputers (free for US researchers).

## The compute landscape for computational scientists

```
TIER      PLATFORM              COST           BEST FOR
────────  ─────────────────────  ─────────────  ────────────────────────────────────────
Free      Google Colab (free)    $0             Prototyping, ML training, GROMACS
Free      Kaggle Kernels         $0             Data science, ML, 2× T4 GPUs
Free      NSF ACCESS Explore     $0 (credits)   SLURM HPC, MD, docking, any research
Free      DOE INCITE / NERSC     $0 (apply)     Petascale for large projects
Low-cost  Google Colab Pro       ~$10/mo        A100 GPU, background execution
Low-cost  Lambda Cloud           $1–3/hr        A100/H100 on-demand, no commitment
Low-cost  Vast.ai                $0.30–2/hr     Budget GPU marketplace
Low-cost  RunPod                 $0.40–2/hr     Serverless GPU, fast start
Full      AWS / GCP / Azure      Market rate    Enterprise, compliance, full stack
────────  ─────────────────────  ─────────────  ────────────────────────────────────────
```

## What you will learn

| Section | Platform | Topic |
|---------|----------|-------|
| 1 | Overview + setup | GPU check, environment test |
| 2 | Google Colab | Free T4/A100, GROMACS, PyTorch, Drive persistence |
| 3 | Kaggle Kernels | Dual T4, datasets, GPU ML |
| 4 | NSF ACCESS | Account creation, 4 allocation tiers, SLURM jobs |
| 5 | SLURM job scripts | CPU arrays, GPU jobs, MPI, interactive sessions |
| 6 | File transfer | scp, rsync, Globus, rclone |
| 7 | Containers (Apptainer) | Reproducible HPC environments |
| 8 | Low-cost cloud GPUs | Lambda, Vast.ai, RunPod |
| 9 | Workflow automation | Snakemake, Nextflow on HPC |
| 10 | Cost calculator + decision guide | Choose the right platform |

## Who this tutorial is for

Computational toxicologists, cheminformaticians, and drug discovery scientists who need
to run: GROMACS MD simulations, AutoDock Vina docking campaigns, GNN/QSAR training,
LLM fine-tuning (ToxLLM), UMAP on large compound libraries, or REINVENT4 generative AI.

---
## Section 1 — Environment Check and Platform Overview

This notebook works identically on:
- **Local machine** — CPU only, no extra setup
- **Google Colab** — free T4/A100 GPU
- **Kaggle** — free dual T4
- **NSF ACCESS HPC** — use as a reference; run scripts via SLURM
- **Lambda / Vast.ai** — rented GPU instance

### Quick platform test

The cell below auto-detects your environment and reports GPU, RAM, and storage.

In [ ]:
# ── Section 1: Detect environment and report available resources ──────────────
import os, sys, subprocess, platform

def run(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.DEVNULL).decode().strip()
    except:
        return 'N/A'

# ── Detect which platform we are on ──────────────────────────────────────────
def detect_platform():
    # Google Colab
    try:
        import google.colab
        return 'Google Colab'
    except ImportError:
        pass
    # Kaggle
    if os.path.exists('/kaggle'):
        return 'Kaggle'
    # HPC cluster (PBS/SLURM)
    if os.environ.get('SLURM_JOB_ID') or os.environ.get('PBS_JOBID'):
        return 'HPC Cluster (SLURM/PBS)'
    # Generic Linux
    return f'Linux ({platform.node()})'

platform_name = detect_platform()
print(f'Platform detected: {platform_name}')
print('='*50)

# ── CPU ───────────────────────────────────────────────────────────────────────
ncpu = os.cpu_count()
cpu_model = run("grep 'model name' /proc/cpuinfo | head -1 | cut -d: -f2").strip()
print(f'CPU: {ncpu} cores — {cpu_model[:50]}')

# ── RAM ───────────────────────────────────────────────────────────────────────
mem_raw = run("free -h | awk '/^Mem:/{print $2}'")
print(f'RAM: {mem_raw}')

# ── Storage ───────────────────────────────────────────────────────────────────
disk_raw = run("df -h / | awk 'NR==2{print $4 \" available of \" $2}'")
print(f'Disk: {disk_raw}')

# ── GPU ───────────────────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        n_gpu = torch.cuda.device_count()
        for i in range(n_gpu):
            name = torch.cuda.get_device_name(i)
            vram = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f'GPU {i}: {name}  ({vram:.1f} GB VRAM)')
        print(f'CUDA version: {torch.version.cuda}')
    else:
        print('GPU: Not available (CPU only)')
except ImportError:
    print('GPU: PyTorch not installed')

# ── Platform-specific tips ────────────────────────────────────────────────────
print()
tips = {
    'Google Colab': [
        'To enable GPU: Runtime → Change runtime type → T4/A100',
        'Free tier: ~12h session, T4 GPU',
        'Pro ($10/mo): A100 GPU, longer sessions, background execution',
        'Mount Drive for persistent storage: drive.mount("/content/drive")',
    ],
    'Kaggle': [
        'Free: 30 GPU hrs/week, dual T4 (×2), 16 GB each',
        'Datasets: attach public datasets, no transfer needed',
        'Sessions limited to 12 hours',
        'Enable GPU: Settings → Accelerator → GPU T4 × 2',
    ],
    'HPC Cluster (SLURM/PBS)': [
        'Submit jobs with sbatch myjob.sh',
        'Check queue: squeue -u $USER',
        'Interactive session: srun --pty --ntasks=1 --gpus=1 bash',
        'Load modules: module avail, module load python/3.10',
    ],
}
if platform_name in tips:
    print(f'Tips for {platform_name}:')
    for t in tips[platform_name]: print(f'  • {t}')
else:
    print('Generic Linux environment. See sections below for platform setup.')

---
## Section 2 — Google Colab: Free GPU for Computational Science

Google Colaboratory is the **fastest way to start computing** — no account
setup beyond a Google login, GPU available in two clicks.

### Free vs Pro comparison

| Feature | Free Colab | Colab Pro (~$10/mo) | Colab Pro+ (~$50/mo) |
|---------|-----------|---------------------|----------------------|
| GPU | T4 (16 GB) | A100 (40 GB) | A100 priority |
| Session limit | ~12 hours | ~24 hours | Background execution |
| RAM | ~13 GB | ~52 GB | ~52 GB |
| Disk | ~77 GB | ~166 GB | ~166 GB |
| Idle timeout | ~90 min | ~3 hours | Extended |

### Enabling GPU (critical first step)

```
Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save
```

### The Colab persistence problem and solution

```
Problem: Colab resets when the session ends — all installed packages and files are lost.

Solution A: Mount Google Drive → files persist permanently
Solution B: Save outputs to Drive at end of each run
Solution C: pip install at the top of every notebook (fast, <2 min)
Solution D: Use Colab Pro + background execution for long runs
```

In [ ]:
# ── Section 2: Google Colab — complete setup code ────────────────────────────

COLAB_SETUP_CODE = '''
# ════════════════════════════════════════════════════════════════════════
# STEP 0: Run this in Google Colab FIRST
# Runtime → Change runtime type → T4 GPU → Save
# ════════════════════════════════════════════════════════════════════════

# ── Step 1: Mount Google Drive (for persistence) ──────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
WORKDIR = '/content/drive/MyDrive/comp_tox_workspace'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')

# ── Step 2: Verify GPU ────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ── Step 3: Install computation stack ────────────────────────────────
# Run once per session (or save to Drive)
!pip install -q rdkit umap-learn scikit-learn imbalanced-learn peft trl
!pip install -q bitsandbytes transformers datasets accelerate

# ── Step 4: Install GROMACS (for MD simulations) ─────────────────────
# Method A: apt-get (fast, pre-compiled, GPU-enabled version)
!apt-get -qq install gromacs
!gmx --version 2>&1 | head -3

# Method B: Pre-compiled binary from Drive (if saved previously)
# GROMACS_BIN = f'{WORKDIR}/gromacs_2024_gpu'
# if os.path.exists(GROMACS_BIN):
#     !{GROMACS_BIN}/bin/gmx --version

# ── Step 5: Quick benchmark ───────────────────────────────────────────
import torch, time
x = torch.randn(10000, 10000, device='cuda')
t0 = time.time()
for _ in range(100):
    y = x @ x.T
torch.cuda.synchronize()
elapsed = time.time() - t0
print(f'GPU matmul benchmark: {100/elapsed:.0f} ops/sec (higher = better)')
'''

print('Google Colab setup code:')
print(COLAB_SETUP_CODE)

# ── Colab GROMACS MD example ──────────────────────────────────────────────────
COLAB_GROMACS = '''
# ═══════════════════════════════════════════════════════════════════════
# Running GROMACS on Colab — Protein MD simulation
# ═══════════════════════════════════════════════════════════════════════

# Download a small test protein
!wget -q https://files.rcsb.org/download/1AKI.pdb

# Prepare topology
!echo 15 | gmx pdb2gmx -f 1AKI.pdb -o protein.gro -water spce -ff oplsaa

# Define simulation box
!gmx editconf -f protein.gro -o box.gro -c -d 1.0 -bt cubic

# Solvate
!gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top

# Add ions
!gmx grompp -f ions.mdp -c solvated.gro -p topol.top -o ions.tpr
!echo SOL | gmx genion -s ions.tpr -o ions.gro -p topol.top -pname NA -nname CL -neutral

# Energy minimisation
!gmx grompp -f em.mdp -c ions.gro -p topol.top -o em.tpr
!gmx mdrun -v -deffnm em -ntmpi 1 -ntomp 2    # CPU-based on Colab free tier
# For GPU: add -gpu_id 0

# Copy output to Drive (persistent!)
import shutil
shutil.copy('em.gro', f'{WORKDIR}/em.gro')
shutil.copy('em.edr', f'{WORKDIR}/em.edr')
print('Saved to Drive — safe after session ends')
'''

print('\nGROMACS on Colab workflow:')
print(COLAB_GROMACS)

In [ ]:
# ── 2.2 Colab ML / LLM fine-tuning pattern ────────────────────────────────────
COLAB_ML_CODE = '''
# ═══════════════════════════════════════════════════════════════════════
# LLM fine-tuning on Colab A100 (Pro)
# or ML training on Colab T4 (free)
# ═══════════════════════════════════════════════════════════════════════

# Check available VRAM and pick model size accordingly
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 40:    # A100 (Pro)
    MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'
    USE_4BIT = True   # still use QLoRA for efficiency
elif vram_gb >= 16:  # T4 (free)
    MODEL = 'microsoft/phi-2'
    USE_4BIT = True
else:
    MODEL = 'microsoft/phi-1_5'
    USE_4BIT = False

print(f'VRAM: {vram_gb:.1f} GB → Using model: {MODEL}')

# QLoRA fine-tuning (standard from our toxllm_tutorial)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
) if USE_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto"
)
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e9:.1f}B params')
# ... continue with SFTTrainer as in toxllm_tutorial.ipynb
'''

print('Colab ML/LLM pattern:')
print(COLAB_ML_CODE)

# ── Colab time-limit workaround ───────────────────────────────────────────────
ANTI_IDLE = '''
# ── Anti-idle trick: keep Colab alive ─────────────────────────────────────
# Paste in browser console (F12 → Console) to prevent auto-disconnect:
# function ClickConnect() {
#     console.log("Staying alive...");
#     document.querySelector("#top-toolbar > colab-connect-button").click()
# }
# setInterval(ClickConnect, 60000)

# Better solution: save checkpoints to Drive every N steps
# trainer = SFTTrainer(
#     ...args=TrainingArguments(save_steps=50, output_dir=WORKDIR)...
# )
'''
print('\nAnti-idle + checkpoint strategy:')
print(ANTI_IDLE)

---
## Section 3 — Kaggle Kernels: Free Dual T4 GPUs

Kaggle gives **30 GPU hours/week for free**, with access to dual T4 GPUs
(2× 16 GB = 32 GB combined), and a huge library of pre-attached datasets.

### Key advantages over Colab

| Feature | Colab Free | Kaggle Free |
|---------|-----------|-------------|
| GPU VRAM | 16 GB (T4 × 1) | 32 GB (T4 × 2) |
| GPU hours/week | Unmetered (but throttled) | 30 hrs/week (strict) |
| Dataset access | Manual upload | One-click attach |
| Internet access | Always on | On by default |
| Reproducibility | Share link | Persistent kernel versions |

### Enabling GPU on Kaggle
```
Settings panel (right side) → Accelerator → GPU T4 × 2 → Save
```

In [ ]:
# ── Section 3: Kaggle setup and DataParallel multi-GPU pattern ───────────────
import os

KAGGLE_SETUP = '''
# ═══════════════════════════════════════════════════════════════════════
# Kaggle kernel setup
# ═══════════════════════════════════════════════════════════════════════

import os, torch

# Check GPUs (should show 2× T4)
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

# Kaggle environment variables
print('Input dir:', os.environ.get('KAGGLE_DATA_PROXY_PROJECT', '/kaggle/input'))
print('Working dir: /kaggle/working')

# ── Load a Kaggle dataset (attached in Settings → Add data) ──────────────
# Example: ChEMBL compound activity dataset
# import pandas as pd
# df = pd.read_csv('/kaggle/input/chembl-activity-data/chembl_activity.csv')

# ── Multi-GPU with DataParallel ───────────────────────────────────────────
import torch
import torch.nn as nn

class ToxModel(nn.Module):
    def __init__(self, in_dim=2048, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, 128),    nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x)

model = ToxModel()

# Wrap with DataParallel for dual-GPU Kaggle
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs with DataParallel')
    model = nn.DataParallel(model)

model = model.cuda()
print(f'Model on: {next(model.parameters()).device}')

# Effective batch size = batch_size × n_gpus (automatic with DataParallel)
# With 2× T4: batch_size=128 → effectively 256 samples per step
'''

print('Kaggle dual-GPU setup:')
print(KAGGLE_SETUP)

# ── Kaggle dataset access patterns ────────────────────────────────────────────
print('Key Kaggle dataset paths for computational toxicology:')
kaggle_datasets = [
    ('/kaggle/input/tox21',           'Tox21 challenge (12 tasks, 8k compounds)'),
    ('/kaggle/input/herg-central',    'hERG cardiotoxicity dataset'),
    ('/kaggle/input/chembl-activity', 'ChEMBL IC50/EC50 activity data'),
    ('/kaggle/input/sider',           'SIDER drug side effects'),
    ('/kaggle/input/clintox',         'ClinTox clinical trial toxicity'),
]
for path, desc in kaggle_datasets:
    print(f'  {path:40s}: {desc}')

---
## Section 4 — NSF ACCESS: Free National Supercomputers for US Researchers

**NSF ACCESS** (Advanced Cyberinfrastructure Coordination Ecosystem:
Services & Support) provides **free HPC time** to US researchers at academic
and non-profit institutions.

### The four allocation tiers (2025)

| Tier | Credits | Review | Best for | Timeline |
|------|---------|--------|----------|----------|
| **Explore** | ≤ 400k credits | Same-day auto | Prototyping, courses, new users | <1 day |
| **Discover** | ≤ 1.5M credits | Expert review | Small–medium projects | 1–2 weeks |
| **Accelerate** | ≤ 3M credits | Panel review | Major projects | 1–3 months |
| **Maximize** | Unlimited | Semi-annual | Petascale, flagship | 3–6 months |

**For a BHSAI/HJF or UMB researcher: Explore is immediately actionable.**
File the request at https://allocations.access-ci.org — approval is same-day.

### Systems available (2025)

| System | Location | Compute | GPUs | Best for |
|--------|----------|---------|------|----------|
| **Expanse** | SDSC, San Diego | AMD Rome, 128c/node | V100 (32 GB) | General HPC, ML |
| **Bridges-2** | PSC, Pittsburgh | 256 GB–4 TB/node | V100/A100 | Big-memory, AI |
| **Stampede3** | TACC, Austin | Intel SPR, 96c/node | H100, PVC | Large simulations |
| **Anvil** | Purdue | AMD Milan, 128c/node | A100 (40 GB) | ML/DL, datasets |
| **Delta** | NCSA, Illinois | AMD Milan | A100, H100 | GPU-heavy workloads |
| **Jetstream2** | Indiana | Cloud VMs | A100 | Interactive, cloud |

### Eligibility note
PI must be affiliated with a US academic or non-profit institution.
Graduate students and postdocs can be PI. Must use an institutional email.
BHSAI/HJF affiliation through UMB qualifies.

In [ ]:
# ── Section 4: NSF ACCESS account creation and allocation workflow ────────────
import os

print('NSF ACCESS Account Setup — Step by Step')
print('='*55)

access_steps = [
    ('Step 1', 'Create ACCESS account',
     'Go to https://allocations.access-ci.org\n'
     '   Click "Register" → use your institutional email\n'
     '   Complete identity verification (ORCID or InCommon)'),
    ('Step 2', 'Log in to ACCESS portal',
     'URL: https://allocations.access-ci.org\n'
     '   Navigate to "Submit a Request" → Explore\n'
     '   Or: click "New Project" under your dashboard'),
    ('Step 3', 'Submit Explore allocation',
     'Project title: e.g. "Computational Toxicology QSAR Models"\n'
     '   Description: 1-2 paragraphs of research goals\n'
     '   Select resource: Expanse (SDSC) or Bridges-2 (PSC)\n'
     '   Request: 400,000 credits (Explore maximum)\n'
     '   Approval: usually same day or next business day'),
    ('Step 4', 'Link allocation to a cluster',
     'After approval, go to resource portal (e.g. XSEDE User Portal)\n'
     '   Add users to your project\n'
     '   SSH keys: add your public key to the portal'),
    ('Step 5', 'SSH to the cluster',
     'Expanse:  ssh username@login.expanse.sdsc.edu\n'
     '   Bridges-2: ssh username@bridges2.psc.edu\n'
     '   Stampede3: ssh username@stampede3.tacc.utexas.edu'),
]

for step, title, detail in access_steps:
    print(f'\n{step}: {title}')
    for line in detail.split('\n'):
        print(f'   {line}')

# ── Credit conversion (ACCESS Credits → SUs) ─────────────────────────────────
print()
print('Credit conversion rates (approximate, 2025):')
credit_table = [
    ('Expanse CPU',       1.0,   '1 credit = 1 CPU-core-hour'),
    ('Expanse GPU',       1.0,   '1 GPU-hour ≈ 1 credit'),
    ('Bridges-2 RM',      1.0,   '1 credit = 1 core-hour'),
    ('Bridges-2 GPU',     1.0,   '1 V100-hour ≈ 1 credit'),
    ('Stampede3 SKX',     1.0,   '1 credit = 1 core-hour'),
    ('Stampede3 H100',    1.0,   '1 H100-hour ≈ 1 credit'),
    ('Anvil CPU',         1.0,   '1 credit = 1 core-hour'),
]
for sys_name, rate, note in credit_table:
    print(f'  {sys_name:25s}: {note}')

print()
print('With 400k Explore credits (maximum):')
estimates = [
    ('GROMACS 20ns protein MD',  '128 CPU-cores × 8h = 1,024 credits → 390 runs possible'),
    ('AutoDock Vina 1000 cpds',  '32 cores × 4h = 128 credits → 3,125 campaigns'),
    ('GNN training (V100)',       '1 GPU × 8h = 8 credits → 50,000 GPU-hours'),
    ('LLM fine-tuning (A100)',    '1 GPU × 12h = 12 credits → 33,333 fine-tuning jobs'),
]
for task, est in estimates:
    print(f'  {task:35s}: {est}')

---
## Section 5 — SLURM Job Scripts for Computational Toxicology

All NSF ACCESS clusters use **SLURM** (Simple Linux Utility for Resource Management).
You submit a shell script that tells SLURM what resources you need and what to run.

### Essential SLURM commands

```bash
sbatch myjob.sh           # submit a job
squeue -u $USER           # check your running/pending jobs
scancel <jobid>           # cancel a job
sacct -j <jobid>          # accounting after job finishes
sinfo                     # see available partitions / node state
srun --pty bash           # interactive session (see below)
```

### SLURM header anatomy

```bash
#!/bin/bash
#SBATCH --job-name=my_job       # name in queue
#SBATCH --account=abc123        # your ACCESS allocation ID
#SBATCH --partition=compute     # partition (queue) name
#SBATCH --nodes=1               # number of compute nodes
#SBATCH --ntasks-per-node=128   # MPI tasks per node
#SBATCH --cpus-per-task=1       # CPU threads per task
#SBATCH --mem=256G              # RAM per node
#SBATCH --time=24:00:00         # max wall time HH:MM:SS
#SBATCH --output=job_%j.out     # stdout log (%j = job ID)
#SBATCH --error=job_%j.err      # stderr log
#SBATCH --mail-type=END,FAIL    # email on finish or failure
#SBATCH --mail-user=your@email  # your email
```

In [ ]:
# ── Section 5: SLURM job scripts for computational toxicology tasks ───────────
import os

os.makedirs('cloud_hpc_scripts', exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# Script 1: GROMACS MD simulation (CPU, Expanse)
# ════════════════════════════════════════════════════════════════════════
slurm_gromacs = '''#!/bin/bash
#SBATCH --job-name=gromacs_md
#SBATCH --account=YOUR_ACCESS_ALLOCATION
#SBATCH --partition=compute              # Expanse partition
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=128            # full Expanse node = 128 cores
#SBATCH --cpus-per-task=1
#SBATCH --mem=256G
#SBATCH --time=24:00:00
#SBATCH --output=gromacs_%j.out
#SBATCH --error=gromacs_%j.err

# ── Environment ──────────────────────────────────────────────────────────
module purge
module load gromacs/2024.1          # check: module avail gromacs

cd $SLURM_SUBMIT_DIR
echo "Job $SLURM_JOB_ID started on $(hostname) at $(date)"

# ── Energy minimisation ───────────────────────────────────────────────────
gmx grompp -f em.mdp -c solvated.gro -p topol.top -o em.tpr
gmx mdrun -v -deffnm em -ntmpi 1 -ntomp $SLURM_CPUS_PER_TASK

# ── NVT equilibration ─────────────────────────────────────────────────────
gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr
gmx mdrun -deffnm nvt -ntmpi 1 -ntomp $SLURM_CPUS_PER_TASK

# ── NPT equilibration ─────────────────────────────────────────────────────
gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr
gmx mdrun -deffnm npt -ntmpi 1 -ntomp $SLURM_CPUS_PER_TASK

# ── Production MD ─────────────────────────────────────────────────────────
gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr
gmx mdrun -deffnm md -ntmpi $SLURM_NTASKS_PER_NODE -ntomp 1

echo "Job completed at $(date)"
'''

with open('cloud_hpc_scripts/slurm_gromacs.sh', 'w') as f:
    f.write(slurm_gromacs)
print('Saved: cloud_hpc_scripts/slurm_gromacs.sh')

# ════════════════════════════════════════════════════════════════════════
# Script 2: GPU GNN/ML training (Expanse GPU partition)
# ════════════════════════════════════════════════════════════════════════
slurm_gnn = '''#!/bin/bash
#SBATCH --job-name=gnn_training
#SBATCH --account=YOUR_ACCESS_ALLOCATION
#SBATCH --partition=gpu-shared              # GPU partition
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=10
#SBATCH --gpus=1                            # 1 × V100 32GB
#SBATCH --mem=96G
#SBATCH --time=12:00:00
#SBATCH --output=gnn_%j.out

module purge
module load gpu/0.15.4  cuda/11.6.2  python/3.9.5

# Activate your virtual environment
source ~/envs/gnn_tox/bin/activate

cd $SLURM_SUBMIT_DIR
echo "GPU: $(nvidia-smi --query-gpu=name --format=csv,noheader)"

# Run your GNN training script
python gnn_toxicology_train.py \
    --data_path data/ames_train.csv \
    --model_type AttentiveFP \
    --epochs 100 \
    --batch_size 256 \
    --output_dir results/
'''

with open('cloud_hpc_scripts/slurm_gnn.sh', 'w') as f:
    f.write(slurm_gnn)
print('Saved: cloud_hpc_scripts/slurm_gnn.sh')

# ════════════════════════════════════════════════════════════════════════
# Script 3: AutoDock Vina array job (1 job per compound)
# ════════════════════════════════════════════════════════════════════════
slurm_vina = '''#!/bin/bash
#SBATCH --job-name=vina_screen
#SBATCH --account=YOUR_ACCESS_ALLOCATION
#SBATCH --partition=compute
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=16G
#SBATCH --time=04:00:00
#SBATCH --output=vina_%A_%a.out
#SBATCH --array=1-1000%50          # 1000 compounds, 50 running simultaneously

module load vina/1.2.5

# Each array task processes one compound
COMPOUND=$(sed -n "${SLURM_ARRAY_TASK_ID}p" compound_list.txt)
echo "Docking compound: $COMPOUND"

vina --receptor receptor.pdbqt \
     --ligand ligands/${COMPOUND}.pdbqt \
     --config docking_config.txt \
     --out results/${COMPOUND}_docked.pdbqt \
     --log results/${COMPOUND}_vina.log \
     --cpu $SLURM_CPUS_PER_TASK
'''

with open('cloud_hpc_scripts/slurm_vina_array.sh', 'w') as f:
    f.write(slurm_vina)
print('Saved: cloud_hpc_scripts/slurm_vina_array.sh')

# ════════════════════════════════════════════════════════════════════════
# Script 4: LLM fine-tuning on Bridges-2 GPU (A100)
# ════════════════════════════════════════════════════════════════════════
slurm_llm = '''#!/bin/bash
#SBATCH --job-name=toxllm_finetune
#SBATCH --account=YOUR_ACCESS_ALLOCATION
#SBATCH --partition=GPU-shared               # Bridges-2 GPU partition
#SBATCH --nodes=1
#SBATCH --gpus=v100-32:1                     # 1 V100 32GB
# For A100: --gpus=a100-40:1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=5
#SBATCH --mem=40G
#SBATCH --time=08:00:00
#SBATCH --output=toxllm_%j.out

module purge
module load AI/pytorch-2.1.0_CUDA-11.8

source ~/envs/toxllm/bin/activate

echo "GPU: $(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader)"
echo "Starting ToxLLM fine-tuning at $(date)"

python toxllm_finetune.py \
    --model mistralai/Mistral-7B-Instruct-v0.3 \
    --train_data toxllm_data/train.jsonl \
    --val_data toxllm_data/validation.jsonl \
    --output_dir $SCRATCH/toxllm_checkpoints \
    --epochs 3

echo "Fine-tuning completed at $(date)"
'''

with open('cloud_hpc_scripts/slurm_toxllm.sh', 'w') as f:
    f.write(slurm_llm)
print('Saved: cloud_hpc_scripts/slurm_toxllm.sh')

# ════════════════════════════════════════════════════════════════════════
# Script 5: Interactive GPU session
# ════════════════════════════════════════════════════════════════════════
interactive_cmd = '''
# ── Interactive GPU session (run on login node) ─────────────────────────
# Expanse GPU node:
srun --partition=gpu-shared \
     --account=YOUR_ALLOCATION \
     --nodes=1 --ntasks=1 --cpus-per-task=8 \
     --gpus=1 --mem=32G --time=02:00:00 \
     --pty /bin/bash

# Bridges-2 GPU node:
srun --partition=GPU-shared \
     --account=YOUR_ALLOCATION \
     --gpus=v100-32:1 \
     --ntasks=1 --cpus-per-task=5 \
     --mem=32G --time=01:00:00 \
     --pty /bin/bash
'''

print('\nInteractive session command:')
print(interactive_cmd)
print()
print('All SLURM scripts saved to cloud_hpc_scripts/')

---
## Section 6 — File Transfer: Getting Data In and Out of HPC

Transferring large datasets efficiently is critical. The right tool
depends on file size, speed requirements, and whether you need automation.

| Tool | Best for | Speed | Setup |
|------|----------|-------|-------|
| `scp` | Small files (<1 GB), one-off | Moderate | Zero |
| `rsync` | Syncing directories, resumable | Good | Zero |
| **Globus** | Large data (>10 GB), lab-to-HPC | Excellent | ~10 min |
| `rclone` | Cloud storage (S3, Drive, Box) | Good | ~5 min |
| `sftp` | Interactive browsing | Moderate | Zero |

### Golden rule: never transfer to/from login nodes

```bash
# BAD: Direct scp to login node — interrupts other users
scp bigfile.tar.gz username@stampede3.tacc.utexas.edu:~/

# GOOD: Use a dedicated data transfer node (DTN) or Globus
scp bigfile.tar.gz username@data.expanse.sdsc.edu:~/  # Expanse DTN
scp bigfile.tar.gz username@data.bridges2.psc.edu:~/  # Bridges-2 DTN
```

In [ ]:
# ── Section 6: File transfer scripts and Globus setup ────────────────────────
import os

# ════════════════════════════════════════════════════════════════════════
# Tool 1: scp / rsync — for files <10 GB
# ════════════════════════════════════════════════════════════════════════
scp_examples = '''
# ── scp: secure copy ─────────────────────────────────────────────────────

# Upload single file to Expanse
scp dataset.csv username@data.expanse.sdsc.edu:/expanse/lustre/scratch/username/myproject/

# Download results from Expanse
scp username@data.expanse.sdsc.edu:/expanse/lustre/scratch/username/myproject/results.tar.gz .

# Upload entire directory
scp -r ./toxllm_data/ username@bridges2.psc.edu:/ocean/projects/YOUR_PROJECT/username/

# ── rsync: the better choice for directories ──────────────────────────────
# Flags: -a (archive), -v (verbose), -z (compress), --progress

# Upload project directory (only transfers new/changed files)
rsync -avz --progress ./comp_tox_project/ \
    username@data.expanse.sdsc.edu:/expanse/lustre/scratch/username/comp_tox/

# Download results, excluding large trajectory files
rsync -avz --exclude='*.xtc' --exclude='*.trr' \
    username@data.expanse.sdsc.edu:/expanse/lustre/scratch/username/comp_tox/results/ \
    ./results_from_hpc/

# Resume interrupted transfer
rsync -avz --partial --progress ./large_file.tar.gz \
    username@data.bridges2.psc.edu:/ocean/projects/PROJECT/username/
'''

print('scp and rsync examples:')
print(scp_examples)

# ════════════════════════════════════════════════════════════════════════
# Tool 2: Globus — for large data transfers (>10 GB)
# ════════════════════════════════════════════════════════════════════════
globus_setup = '''
# ── Globus: fastest method for large transfers ────────────────────────────
# Speed: up to 10+ GB/s between HPC endpoints
# Free for academic use: https://www.globus.org

# Step 1: Install Globus Connect Personal on your laptop
#   macOS/Windows/Linux: https://www.globus.org/globus-connect-personal
#   This makes your laptop a Globus endpoint

# Step 2: Log in at https://app.globus.org
#   Use your institutional login (ACCESS, InCommon, Google)

# Step 3: In the File Manager, set two endpoints:
#   Left panel:  your laptop (Globus Connect Personal endpoint)
#   Right panel: search for the HPC system:
#     Expanse:   'SDSC Expanse'
#     Bridges-2: 'Pittsburgh Supercomputing Center'  
#     Stampede3: 'TACC Stampede3'
#     Anvil:     'Purdue Anvil'

# Step 4: Navigate to your paths and click Transfer
#   HPC paths:
#     Expanse:   /expanse/lustre/scratch/USERNAME/
#     Bridges-2: /ocean/projects/PROJECT_ID/USERNAME/
#     Stampede3: /scratch/00000/USERNAME/

# ── Globus CLI (command line) ─────────────────────────────────────────────
# pip install globus-cli
# globus login

# Find endpoint IDs
# globus endpoint search 'SDSC Expanse'

# Transfer files from command line
# SRC_ID='aa1d3f14-b89c-11e7-afd2-22000b9a448b'  # your laptop endpoint
# DST_ID='4bb864a8-7b44-11e9-bbf7-0a37f382de32'  # Expanse endpoint
#
# globus transfer \
#   ${SRC_ID}:/path/to/local/data \
#   ${DST_ID}:/expanse/lustre/scratch/username/data \
#   --recursive --label 'comp_tox_data_upload'
'''

print('Globus setup and usage:')
print(globus_setup)

# ════════════════════════════════════════════════════════════════════════
# Tool 3: rclone — for cloud storage (Google Drive, S3, Box)
# ════════════════════════════════════════════════════════════════════════
rclone_setup = '''
# ── rclone: sync with cloud storage ──────────────────────────────────────
# Install: pip install rclone  (or: curl -fsSL https://rclone.org/install.sh | bash)

# Step 1: Configure a remote (one-time)
# rclone config
# → n (new remote) → name: gdrive → type: drive → follow OAuth prompts

# Upload results from HPC to Google Drive
# rclone copy /expanse/lustre/scratch/username/results/ gdrive:hpc_results/

# Download large dataset from S3 to HPC
# rclone copy s3:my-bucket/compound_library/ /expanse/lustre/scratch/username/data/

# Sync local and remote (bidirectional)
# rclone sync gdrive:comp_tox/ /expanse/lustre/scratch/username/comp_tox/ -v
'''

print('rclone cloud storage setup:')
print(rclone_setup)

# Save transfer cheatsheet
transfer_cheat = '''
FILE TRANSFER QUICK REFERENCE

Small files (<1 GB):   scp file.csv user@data.expanse.sdsc.edu:/path/
Directories:           rsync -avz --progress ./dir/ user@data.expanse.sdsc.edu:/path/
Large data (>10 GB):   Globus web UI at app.globus.org
Cloud storage:         rclone copy /local/path gdrive:remote/path

HPC Data Transfer Nodes (use these, not login nodes):
  Expanse:   data.expanse.sdsc.edu
  Bridges-2: data.bridges2.psc.edu
  Stampede3: stampede3.tacc.utexas.edu (standard login is fine for data)
  Anvil:     anvil.rcac.purdue.edu
'''

with open('cloud_hpc_scripts/file_transfer_cheatsheet.txt', 'w') as f:
    f.write(transfer_cheat)
print('Saved: cloud_hpc_scripts/file_transfer_cheatsheet.txt')

---
## Section 7 — Containers on HPC: Apptainer (Singularity)

**Why containers?** HPC systems have fixed module versions. Your code
may need a different Python version, a specific PyTorch build, or
software not installed on the system (e.g. REINVENT4).

**Docker is not allowed on HPC** (security: root access required).
**Apptainer** (formerly Singularity) is the HPC-standard container system.

```
Docker image (on your laptop/DockerHub)
       │
       │  apptainer pull / build
       ▼
.sif file (Singularity Image File)
       │
       │  apptainer exec container.sif python script.py
       ▼
Your code runs inside the container on the HPC node
```

In [ ]:
# ── Section 7: Apptainer containers on HPC ───────────────────────────────────

container_code = '''
# ═══════════════════════════════════════════════════════════════════════
# Apptainer (Singularity) container workflow for HPC
# ═══════════════════════════════════════════════════════════════════════

# ── Step 1: Build on your laptop (needs root), then copy to HPC ─────────
# Or: use a cloud VM to build, then copy .sif to HPC

# Apptainer definition file: toxllm.def
# ────────────────────────────────────────────────────────────────────────
# Bootstrap: docker
# From: pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime
#
# %post
#     pip install --no-cache-dir transformers peft trl bitsandbytes
#     pip install --no-cache-dir datasets accelerate sentencepiece
#     pip install --no-cache-dir rdkit umap-learn scikit-learn imbalanced-learn
#
# %runscript
#     python "$@"
# ────────────────────────────────────────────────────────────────────────

# Build the container (on a machine with root / Docker)
# apptainer build toxllm.sif toxllm.def

# ── Step 2: Upload to HPC ────────────────────────────────────────────────
# rsync -avz toxllm.sif username@data.expanse.sdsc.edu:/expanse/lustre/scratch/username/

# ── Step 3: Run on HPC ───────────────────────────────────────────────────

# Interactive (on compute node):
# apptainer exec --nv toxllm.sif python toxllm_finetune.py
# --nv flag: passes NVIDIA GPU drivers through to container

# ── Step 4: SLURM job using container ────────────────────────────────────
'''

print('Apptainer container workflow:')
print(container_code)

slurm_container = '''#!/bin/bash
#SBATCH --job-name=toxllm_container
#SBATCH --account=YOUR_ALLOCATION
#SBATCH --partition=GPU-shared
#SBATCH --gpus=v100-32:1
#SBATCH --ntasks=1 --cpus-per-task=5 --mem=40G --time=08:00:00
#SBATCH --output=container_%j.out

# Load Apptainer (called Singularity on older clusters)
module load singularity/3.9.0  # or: module load apptainer

CONTAINER=/expanse/lustre/scratch/$USER/toxllm.sif
DATA=/expanse/lustre/scratch/$USER/toxllm_data

# Run Python script inside container with GPU pass-through
apptainer exec --nv \
    --bind $SCRATCH:/scratch \
    --bind $DATA:/data \
    $CONTAINER python /scratch/toxllm_finetune.py \
        --train_data /data/train.jsonl \
        --output_dir /scratch/checkpoints
'''

with open('cloud_hpc_scripts/slurm_container.sh', 'w') as f:
    f.write(slurm_container)
print('Saved: cloud_hpc_scripts/slurm_container.sh')

# ── Pre-built containers for computational science ────────────────────────
print()
print('Ready-to-use containers on DockerHub / NGC:')
containers = [
    ('pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime', 'PyTorch + CUDA (ML/DL)'),
    ('nvcr.io/nvidia/gromacs:2024.1', 'GROMACS with GPU (NVIDIA NGC)'),
    ('biocontainers/autodock-vina:v1.2.5_cv1', 'AutoDock Vina'),
    ('pytorch/pytorch:2.1.0-cuda11.8-cudnn8-devel', 'PyTorch dev build (full)'),
    ('nvcr.io/nvidia/pytorch:24.01-py3', 'NVIDIA optimised PyTorch'),
]
for image, desc in containers:
    print(f'  apptainer pull {image.split("/")[-1].split(":")[0]}.sif docker://{image}')
    print(f'    # {desc}')

---
## Section 8 — Low-Cost Cloud GPUs: Lambda, Vast.ai, RunPod

When you need a GPU **right now** without the allocation application process,
these platforms offer A100/H100 GPUs at a fraction of AWS/Azure pricing.

### 2025 pricing comparison

| Provider | A100 (40 GB) | H100 (80 GB) | Setup | Notes |
|----------|-------------|-------------|-------|-------|
| **Lambda Cloud** | ~$1.29/hr | ~$2.49/hr | 2 min | Reliable, stable |
| **Vast.ai** | ~$0.80/hr | ~$1.80/hr | 5 min | Marketplace, variable |
| **RunPod** | ~$1.19/hr | ~$1.99/hr | 2 min | Fast startup |
| AWS p3.2xlarge | ~$3.06/hr | N/A | 5 min | Enterprise, compliance |
| GCP A100 | ~$3.67/hr | ~$5.00/hr | 5 min | GCP integration |

**Lambda Cloud** is the most popular choice for computational science — stable,
persistent file systems, and Ubuntu environments with CUDA pre-installed.

### When to use paid cloud vs ACCESS HPC

```
Use ACCESS HPC when:              Use paid cloud when:
  Large CPU simulations            You need GPU RIGHT NOW
  MPI multi-node jobs              Interactive debugging
  You can wait in queue            Short burst compute (<24h)
  >1M CPU-core-hours needed        No ACCESS account yet
  Free compute is needed           Proprietary data (private)
```

In [ ]:
# ── Section 8: Low-cost cloud GPU setup and usage ────────────────────────────
import os

# ════════════════════════════════════════════════════════════════════════
# Lambda Cloud: https://lambdalabs.com/cloud
# ════════════════════════════════════════════════════════════════════════
lambda_setup = '''
# ── Lambda Cloud: Step-by-step ────────────────────────────────────────────

# Step 1: Create account at https://lambdalabs.com/cloud
# Step 2: Add payment method
# Step 3: Add SSH key (Settings → SSH Keys → Add SSH Key)
#   ssh-keygen -t ed25519 -C 'lambda_key'
#   cat ~/.ssh/id_ed25519.pub  # paste this into Lambda

# Step 4: Launch instance
#   Dashboard → Launch Instance
#   GPU: NVIDIA A100 SXM4 40 GB  ($1.29/hr)
#   Region: us-east-1 or us-west-1
#   Filesystem: create a persistent filesystem (survives instance termination)
#               Mount path: /home/ubuntu/comp_tox

# Step 5: SSH to instance
# ssh ubuntu@<your-instance-ip>

# Step 6: Environment setup (CUDA already installed)
# pip install torch torchvision  # CUDA version auto-selected
# pip install rdkit transformers peft trl bitsandbytes datasets

# Step 7: Run your computation
# python toxllm_finetune.py

# Step 8: IMPORTANT — terminate instance when done
# Lambda charges by the second. Stop the instance from the dashboard.
# Persistent filesystem is NOT deleted when instance terminates.
'''

print('Lambda Cloud setup:')
print(lambda_setup)

# ════════════════════════════════════════════════════════════════════════
# Vast.ai: GPU marketplace, cheapest option
# ════════════════════════════════════════════════════════════════════════
vast_setup = '''
# ── Vast.ai: GPU marketplace ──────────────────────────────────────────────

# Step 1: Create account at https://vast.ai
# Step 2: Add credit ($10–20 to start)
# Step 3: Search for instances
#   https://cloud.vast.ai/
#   Filter: CUDA 12+, min 40 GB VRAM, DLPerf > 10
#   Sort by: $/hr

# Step 4: Choose Docker image for environment
#   pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime  (for PyTorch)
#   nvidia/cuda:12.1.1-cudnn8-devel-ubuntu22.04     (clean CUDA env)

# Step 5: Set SSH public key in account settings, then:
# ssh -p PORT root@<host>

# Vast.ai CLI (optional):
# pip install vastai
# vastai set api-key YOUR_API_KEY
# vastai search offers 'gpu_name=A100 num_gpus=1 dph<2.0 inet_up>500'
# vastai create instance OFFER_ID --image pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime
'''

print('Vast.ai marketplace setup:')
print(vast_setup)

# ════════════════════════════════════════════════════════════════════════
# Cost estimation for common computational tasks
# ════════════════════════════════════════════════════════════════════════
print()
print('Cost estimates for common computational toxicology tasks:')
print('='*65)
print(f'{"Task":40s} {"Hours":>6} {"GPU":>12} {"Cost":>8}')
print('-'*65)

tasks = [
    ('ToxLLM fine-tuning (Mistral-7B, 3 epochs)', 3.0,  'A100', 1.29),
    ('GNN training (1M molecules, 100 epochs)',    2.0,  'A100', 1.29),
    ('REINVENT4 RL (staged learning, 500 steps)', 1.0,  'V100', 0.80),
    ('UMAP 1M compounds (GPU-accelerated)',        0.5,  'T4',   0.40),
    ('AutoDock-GPU 10k compounds',                 4.0,  'A100', 1.29),
    ('GROMACS 100ns MD (GPU)',                     8.0,  'A100', 1.29),
    ('FEP simulation 10 compounds',                24.0, 'A100', 1.29),
]
for task, hrs, gpu, rate in tasks:
    cost = hrs * rate
    print(f'{task:40s} {hrs:>6.1f}h {gpu:>12s} ${cost:>6.2f}')
print()
print('All prices approximate as of 2025. Check current rates.')

---
## Section 9 — Workflow Automation: Snakemake and Nextflow on HPC

Manual job submission does not scale. When you have **pipelines**
(docking → scoring → ADMET → filtering), automation tools manage
dependencies, restarts, and parallelism automatically.

| Tool | Language | Best for | HPC support |
|------|----------|----------|-------------|
| **Snakemake** | Python-like | Bioinformatics, cheminformatics | SLURM native |
| **Nextflow** | Groovy DSL | Large-scale pipelines | SLURM, AWS, GCP |
| **WDL/Cromwell** | WDL | Genomics (GATK) | HPC + cloud |
| `make` | Makefile | Simple pipelines | Manual |

**Snakemake** is the standard in cheminformatics and computational biology
because its Python-like syntax integrates seamlessly with scientific Python.

In [ ]:
# ── Section 9: Snakemake + Nextflow pipeline examples ────────────────────────
import os

# ════════════════════════════════════════════════════════════════════════
# Snakemake: virtual screening pipeline
# ════════════════════════════════════════════════════════════════════════
snakefile = '''
# Snakefile — Virtual screening pipeline
# pip install snakemake
# Run: snakemake --cores 8 --slurm (on HPC) or snakemake --cores 8 (local)

# Configuration
COMPOUNDS = glob_wildcards('ligands/{compound}.smi').compound
RECEPTOR  = 'receptor/target.pdbqt'

rule all:
    """Final target: all compounds screened and filtered."""
    input:
        'results/final_hits.csv'

rule prepare_ligands:
    """Convert SMILES to PDBQT using Meeko."""
    input:  'ligands/{compound}.smi'
    output: 'pdbqt/{compound}.pdbqt'
    shell:
        'python prepare_ligand.py {input} {output}'

rule dock_compound:
    """Dock one compound with Vina."""
    input:
        ligand='pdbqt/{compound}.pdbqt',
        receptor=RECEPTOR
    output:
        'docked/{compound}_out.pdbqt',
        score='scores/{compound}.txt'
    threads: 8
    shell:
        'vina --receptor {input.receptor} --ligand {input.ligand} '
        '--config docking.conf --out {output[0]} '
        '--log {output.score} --cpu {threads}'

rule compute_admet:
    """Compute ADMET properties for docked compounds."""
    input:  'pdbqt/{compound}.pdbqt'
    output: 'admet/{compound}_admet.json'
    script: 'scripts/compute_admet.py'

rule filter_hits:
    """Aggregate all scores and ADMET, apply cutoffs."""
    input:
        scores=expand('scores/{compound}.txt', compound=COMPOUNDS),
        admet=expand('admet/{compound}_admet.json', compound=COMPOUNDS)
    output: 'results/final_hits.csv'
    script: 'scripts/filter_hits.py'
'''

with open('cloud_hpc_scripts/Snakefile', 'w') as f:
    f.write(snakefile)
print('Saved: cloud_hpc_scripts/Snakefile')

# ── Snakemake SLURM profile ───────────────────────────────────────────────
snakemake_slurm = '''
# Run Snakemake with SLURM backend (submits each rule as a job)

snakemake \
    --slurm \
    --default-resources slurm_account=YOUR_ALLOCATION \
                         slurm_partition=compute \
                         mem_mb=8000 \
                         runtime=120 \
    --jobs 500 \
    --retries 2 \
    all
'''

print()
print('Snakemake + SLURM command:')
print(snakemake_slurm)

# ════════════════════════════════════════════════════════════════════════
# Nextflow: ML training pipeline
# ════════════════════════════════════════════════════════════════════════
nextflow_pipeline = '''
// nextflow_gnn.nf — GNN training and evaluation pipeline
// Run: nextflow run nextflow_gnn.nf -profile slurm

params {
    data_dir   = './data'
    output_dir = './results'
    epochs     = 100
    models     = ['GCN', 'GAT', 'AttentiveFP']
}

process PREPARE_FINGERPRINTS {
    conda 'rdkit'
    input:  path smiles_csv
    output: path 'fingerprints.npy'
    script:
    '''
    python compute_fingerprints.py --input $smiles_csv --output fingerprints.npy
    '''
}

process TRAIN_GNN {
    label 'gpu'
    input:  each model from params.models; path fp from PREPARE_FINGERPRINTS.out
    output: path "${model}_checkpoint.pt"
    script:
    '''
    python train_gnn.py --model !{model} --data !{fp} --epochs !{params.epochs}
    '''
}

process EVALUATE {
    input:  path checkpoints from TRAIN_GNN.out.collect()
    output: path 'benchmark_results.csv'
    script:
    '''
    python evaluate_models.py --checkpoints $checkpoints
    '''
}

workflow {
    PREPARE_FINGERPRINTS(file(params.data_dir + '/compounds.csv'))
    TRAIN_GNN(PREPARE_FINGERPRINTS.out)
    EVALUATE(TRAIN_GNN.out.collect())
}
'''

with open('cloud_hpc_scripts/nextflow_gnn.nf', 'w') as f:
    f.write(nextflow_pipeline)
print('Saved: cloud_hpc_scripts/nextflow_gnn.nf')

---
## Section 10 — Cost Calculator, Decision Guide, and Complete Reference

### Decision flowchart

```
Do you need a GPU?
  │
  ├── No (CPU-only) → NSF ACCESS Explore (free, 400k credits)
  │
  └── Yes
       │
       ├── Can you wait 1-3 days for queue?
       │     └── Yes → NSF ACCESS (free, GPU partitions)
       │
       └── Need GPU now?
             │
             ├── ≤12 hours, no persistence needed → Colab (free T4/A100)
             ├── ≤30 hrs/week, large data → Kaggle (free, dual T4)
             └── More than that → Lambda / Vast.ai / RunPod (~$1-3/hr)
```

In [ ]:
# ── Section 10: Cost calculator and decision visualisation ───────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── 10.1 Cost comparison across platforms ────────────────────────────────────
PLATFORMS = {
    # name: (hourly_cost_USD, vram_GB, n_gpus, queue_wait_hours, notes)
    'Colab Free (T4)':    (0.0,    16,  1, 0.0, 'Free, sessions up to 12h'),
    'Colab Pro (A100)':   (10/720, 40,  1, 0.0, '~$10/mo, A100, longer sessions'),
    'Kaggle (T4×2)':      (0.0,    32,  2, 0.0, 'Free, 30 hrs/week limit'),
    'NSF ACCESS Explore': (0.0,    32,  1, 4.0, 'Free, Explore tier, queue wait'),
    'NSF ACCESS Discover':(0.0,    80,  1, 12.0,'Free, larger jobs, longer wait'),
    'Vast.ai A100':       (0.80,   40,  1, 0.0, 'Marketplace, variable reliability'),
    'RunPod A100':        (1.19,   40,  1, 0.0, 'Fast start, reliable'),
    'Lambda A100':        (1.29,   40,  1, 0.0, 'Most reliable, persistent FS'),
    'Lambda H100':        (2.49,   80,  1, 0.0, 'Best performance'),
    'AWS p3.2xl (V100)':  (3.06,   16,  1, 0.0, 'Enterprise, compliance'),
    'GCP A100':           (3.67,   40,  1, 0.0, 'Google Cloud integration'),
}

# Representative jobs with GPU-hours needed
JOBS = {
    'ToxLLM fine-tuning (Mistral-7B, 3 epochs)':  3.0,
    'GNN training (300k mol, 100 ep)':             2.0,
    'REINVENT4 RL (500 steps)':                    1.0,
    'GROMACS 100ns MD simulation':                 8.0,
    'LLM fine-tuning (7B, full job)':             12.0,
    'AutoDock-GPU 50k compounds':                  6.0,
}

print('Cost per job across platforms (USD):')
print(f'{"Job":45s}', end='')
short_names = {
    'Colab Free (T4)': 'ColabFr',
    'Colab Pro (A100)': 'ColabPr',
    'Kaggle (T4×2)':    'Kaggle',
    'NSF ACCESS Explore':'ACCESS',
    'Vast.ai A100':     'Vast.ai',
    'RunPod A100':      'RunPod',
    'Lambda A100':      'Lambda',
    'Lambda H100':      'LambdaH',
    'AWS p3.2xl (V100)':'AWS',
    'GCP A100':         'GCP',
}
for sn in short_names.values(): print(f'{sn:>8}', end='')
print()
print('-'*125)
for job, gpu_hrs in JOBS.items():
    print(f'{job:45s}', end='')
    for pname, (rate, vram, ngpu, wait, note) in PLATFORMS.items():
        if rate == 0:
            cost_str = 'FREE'
        else:
            cost_str = f'${rate*gpu_hrs:.2f}'
        sn = short_names.get(pname, pname[:7])
        print(f'{cost_str:>8}', end='')
    print()

In [ ]:
# ── 10.2 Dashboard visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.40)
BLUE='#1565C0'; RED='#E74C3C'; GREEN='#27AE60'; GOLD='#F1C40F'; GREY='#95A5A6'

# ── Panel 1: Platform tier overview ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0:2])
ax1.set_xlim(0,12); ax1.set_ylim(0,10); ax1.axis('off')
ax1.set_facecolor('#F8F9FA')
ax1.set_title('Cloud HPC Resource Landscape for Computational Science',
              fontweight='bold', fontsize=12)

tiers = [
    (0.3, 7.5, 3.5, 1.6, 'FREE TIER',   GREEN,  ['Google Colab (T4/A100)', 'Kaggle (T4×2)', 'NSF ACCESS (national HPC)']),
    (4.3, 7.5, 3.5, 1.6, 'LOW-COST',    BLUE,   ['Lambda Cloud ($1-3/hr)', 'Vast.ai ($0.8-2/hr)', 'RunPod ($1.2-2.5/hr)']),
    (8.3, 7.5, 3.5, 1.6, 'ENTERPRISE',  GREY,   ['AWS EC2 ($3-8/hr)', 'Google Cloud ($3-6/hr)', 'Azure ($3-6/hr)']),
    (0.3, 4.5, 3.5, 1.6, 'NATIONAL HPC',GOLD,   ['Expanse (SDSC)', 'Bridges-2 (PSC)', 'Stampede3 (TACC)', 'Anvil (Purdue)']),
    (4.3, 4.5, 3.5, 1.6, 'INTERACTIVE', '#8E44AD', ['Jetstream2 (cloud VM)', 'Colab (notebook)', 'Kaggle (notebook)']),
    (8.3, 4.5, 3.5, 1.6, 'CONTAINERS',  '#2ECC71', ['Apptainer/Singularity', 'Docker → .sif', 'Pre-built NGC images']),
]
for x,y,w,h,label,col,items in tiers:
    r = mpatches.FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.1',
                                 facecolor=col,alpha=0.2,edgecolor=col,lw=2.5)
    ax1.add_patch(r)
    ax1.text(x+w/2, y+h-0.2, label, ha='center', va='top',
             fontsize=10, fontweight='bold', color=col)
    for i, item in enumerate(items):
        ax1.text(x+0.15, y+h-0.6-i*0.3, f'• {item}', va='top', fontsize=8.5)

# ── Panel 2: Cost vs performance scatter ─────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
plat_data = {
    'Colab T4':      (0,  16,  GREEN, 'Free'),
    'Kaggle 2×T4':   (0,  32,  GREEN, 'Free'),
    'ACCESS GPU':    (0,  40,  GOLD,  'Free (queue)'),
    'Vast.ai A100':  (0.80,40, BLUE,  '~$0.80/hr'),
    'Lambda A100':   (1.29,40, BLUE,  '~$1.29/hr'),
    'Lambda H100':   (2.49,80, BLUE,  '~$2.49/hr'),
    'AWS V100':      (3.06,16, GREY,  '~$3.06/hr'),
    'GCP A100':      (3.67,40, GREY,  '~$3.67/hr'),
}
for name, (cost, vram, col, label) in plat_data.items():
    ax2.scatter(cost, vram, s=200, c=col, edgecolors='k', lw=1.5, zorder=5, alpha=0.9)
    ax2.annotate(name, (cost, vram), textcoords='offset points', xytext=(6, 3), fontsize=8)
ax2.set_xlabel('Cost per hour (USD)')
ax2.set_ylabel('GPU VRAM (GB)')
ax2.set_title('Cost vs GPU VRAM\n(bigger/cheaper = better)', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-0.2, 4.2)
ax2.fill_between([0, 0.01], [0, 0], [90, 90], alpha=0.1, color=GREEN, label='Free')
ax2.legend(fontsize=8)

# ── Panel 3: ACCESS allocation tiers ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
tiers_data = ['Explore\n(400k)', 'Discover\n(1.5M)', 'Accelerate\n(3M)', 'Maximize\n(unlimited)']
credits = [400, 1500, 3000, 10000]
wait_days = [0, 14, 60, 180]
bar_cols = [GREEN, BLUE, GOLD, RED]
x = np.arange(len(tiers_data))
ax3b = ax3.twinx()
ax3.bar(x, credits, color=bar_cols, alpha=0.85, edgecolor='white')
ax3b.plot(x, wait_days, 'ko--', lw=2, ms=8, label='Typical wait (days)')
ax3.set_xticks(x); ax3.set_xticklabels(tiers_data, fontsize=9)
ax3.set_ylabel('Max credits (thousands)')
ax3b.set_ylabel('Typical approval time (days)')
ax3.set_title('NSF ACCESS Allocation Tiers', fontweight='bold')
ax3b.legend(fontsize=9, loc='upper left')
ax3.grid(True, alpha=0.3, axis='y')

# ── Panel 4: GPU hours achievable per dollar ──────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
budget_100 = {'Colab Free': 'Unlimited (limited time)', 'Kaggle': '30 hrs/week max',
               'ACCESS Explore': '400k credits', 'Vast.ai': '125 GPU-hrs',
               'Lambda A100': '78 GPU-hrs', 'Lambda H100': '40 GPU-hrs',
               'AWS V100': '33 GPU-hrs'  }
numeric = {'Vast.ai': 125, 'Lambda A100': 78, 'Lambda H100': 40, 'AWS V100': 33}
ax4.barh(list(numeric.keys()), list(numeric.values()),
         color=[BLUE,BLUE,BLUE,GREY], alpha=0.85, edgecolor='white')
ax4.axvline(78, c='k', lw=1.5, ls='--', alpha=0.4)
ax4.set_xlabel('GPU hours for $100')
ax4.set_title('GPU Hours per $100\n(paid platforms only)', fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')
for i, (k, v) in enumerate(numeric.items()):
    ax4.text(v+1, i, str(v), va='center', fontsize=9, fontweight='bold')

# ── Panel 5: File listing table ───────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
files_data = [
    ['cloud_hpc_scripts/slurm_gromacs.sh',     'GROMACS MD SLURM job'],
    ['cloud_hpc_scripts/slurm_gnn.sh',          'GNN GPU training SLURM'],
    ['cloud_hpc_scripts/slurm_vina_array.sh',   'Vina array docking SLURM'],
    ['cloud_hpc_scripts/slurm_toxllm.sh',       'ToxLLM fine-tuning SLURM'],
    ['cloud_hpc_scripts/slurm_container.sh',    'Apptainer container SLURM'],
    ['cloud_hpc_scripts/Snakefile',              'Snakemake pipeline'],
    ['cloud_hpc_scripts/nextflow_gnn.nf',        'Nextflow GNN pipeline'],
    ['cloud_hpc_scripts/file_transfer_cheatsheet.txt', 'Transfer reference'],
]
tbl = ax5.table(cellText=files_data, colLabels=['File','Purpose'],
                cellLoc='left', loc='center', bbox=[0,0.1,1,0.88])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for j in range(2):
    tbl[0,j].set_facecolor(BLUE)
    tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,len(files_data)+1):
    for j in range(2):
        tbl[i,j].set_facecolor('#EBF5FB' if i%2==0 else 'white')
ax5.set_title('Scripts Generated by This Tutorial', fontweight='bold')

fig.suptitle('Cloud and HPC Resources for Computational Science: Complete Overview',
             fontsize=14, fontweight='bold')
plt.savefig('cloud_hpc_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: cloud_hpc_dashboard.png')

In [ ]:
# ── 10.3 Complete cheatsheet + quick-start guide ─────────────────────────────
import os

cheat = [
    'CLOUD AND HPC RESOURCES — COMPLETE REFERENCE',
    '',
    'FREE RESOURCES (start here)',
    '  Google Colab:  colab.research.google.com  — free T4 GPU, 12h sessions',
    '  Colab Pro:     ~$10/mo                    — A100, longer sessions',
    '  Kaggle:        kaggle.com                 — 2×T4, 30 hrs/week',
    '  NSF ACCESS:    allocations.access-ci.org  — national HPC, free for US researchers',
    '    Explore: 400k credits, same-day approval',
    '    Discover: 1.5M credits, 1-2 week review',
    '',
    'NSF ACCESS QUICK START',
    '  1. Register: allocations.access-ci.org → Register (institutional email)',
    '  2. Apply: Submit Request → Explore → describe project → choose Expanse/Bridges-2',
    '  3. SSH: ssh username@login.expanse.sdsc.edu',
    '  4. Load modules: module avail python; module load python/3.10',
    '  5. Submit job: sbatch myjob.sh',
    '  6. Monitor: squeue -u $USER',
    '',
    'SLURM ESSENTIAL COMMANDS',
    '  sbatch myjob.sh              submit job',
    '  squeue -u $USER             check your jobs',
    '  scancel <jobid>              cancel job',
    '  sacct -j <jobid>             job accounting after completion',
    '  sinfo -p gpu                 show GPU node availability',
    '  srun --pty --gpus=1 bash     interactive GPU session',
    '',
    'HPC FILESYSTEM CHEATSHEET',
    '  Expanse:',
    '    $HOME:    /home/username              50 GB, backed up, slow',
    '    $SCRATCH: /expanse/lustre/scratch/user/ 1TB, FAST, not backed up',
    '    Use $SCRATCH for I/O-intensive jobs!',
    '  Bridges-2:',
    '    $HOME:    /home/username              25 GB, backed up',
    '    Projects: /ocean/projects/PROJECT_ID/user/ 2TB shared, fast',
    '',
    'FILE TRANSFER COMMANDS',
    '  Small files: scp file.csv user@data.expanse.sdsc.edu:/expanse/lustre/scratch/user/',
    '  Directories: rsync -avz ./dir/ user@data.bridges2.psc.edu:/ocean/projects/PID/user/',
    '  Large data:  Globus web UI at app.globus.org',
    '  Cloud sync:  rclone copy /local/path gdrive:remote_path',
    '',
    'CONTAINERS ON HPC',
    '  Build: apptainer build myenv.sif myenv.def  (needs root, do locally)',
    '  Pull:  apptainer pull pytorch.sif docker://pytorch/pytorch:2.1.0-cuda11.8',
    '  Run:   apptainer exec --nv pytorch.sif python train.py',
    '  SLURM: module load singularity; apptainer exec --nv container.sif python script.py',
    '',
    'LOW-COST CLOUD GPU (when you need GPU now)',
    '  Lambda:   lambdalabs.com    A100=$1.29/hr  H100=$2.49/hr  (most reliable)',
    '  Vast.ai:  vast.ai           A100~$0.80/hr  (cheapest, marketplace)',
    '  RunPod:   runpod.io         A100=$1.19/hr  (fast start)',
    '',
    'DECISION GUIDE',
    '  CPU-only simulation (GROMACS 1ns) → ACCESS Explore (free)',
    '  GPU ML training <12h             → Colab Pro (A100, ~$0.01)',
    '  GPU training, no queue wanted     → Lambda A100 ($1.29/hr)',
    '  Large array job (1000 compounds)  → ACCESS Discover (free)',
    '  Interactive debugging GPU         → Colab / srun interactive',
    '  Reproducible pipeline             → Snakemake + SLURM',
    '  Petascale (>10M CPU-hours)        → ACCESS Accelerate/Maximize',
]
print('\n'.join(cheat))

# File listing
print()
print('='*60)
print('Scripts created by this tutorial:')
scripts = [
    'cloud_hpc_scripts/slurm_gromacs.sh',
    'cloud_hpc_scripts/slurm_gnn.sh',
    'cloud_hpc_scripts/slurm_vina_array.sh',
    'cloud_hpc_scripts/slurm_toxllm.sh',
    'cloud_hpc_scripts/slurm_container.sh',
    'cloud_hpc_scripts/Snakefile',
    'cloud_hpc_scripts/nextflow_gnn.nf',
    'cloud_hpc_scripts/file_transfer_cheatsheet.txt',
    'cloud_hpc_dashboard.png',
]
for f in scripts:
    status = 'OK' if os.path.exists(f) else '--'
    print(f'  [{status}] {f}')